# 图像卷积

## 互相关运算

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape          # K的高度和宽度
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))       # 输出矩阵的形状
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i : i + h, j : j + w] * K).sum()           # 互相关运算
    return Y

## 验证上述二维互相关运算的输出

In [ ]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

## 实现二维卷积层

In [ ]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

## 卷积层的一个简单应用：检测图像中不同颜色的边缘

In [ ]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

In [ ]:
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)

## 卷积核K只可以检测垂直边缘

In [ ]:
corr2d(X.t(), K)        # 将X转置后，输出为0矩阵

## 学习由X生成Y的卷积核

In [28]:
conv2d = nn.Conv2d(1, 1, kernel_size = (1, 2), bias = False)

X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))

for i in range(10):
        Y_hat = conv2d(X)           # 模型预测
        l = (Y_hat - Y) ** 2        # 均方误差
        conv2d.zero_grad()          # 梯度清零
        l.sum().backward()          # 反向传播
        conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad          # 权重更新
        if (i + 1) % 2 == 0:
            print(f"batch {i + 1}, loss {l.sum():.3f}")         # 偶数批次输出loss值

batch 2, loss 12.930
batch 4, loss 2.171
batch 6, loss 0.365
batch 8, loss 0.062
batch 10, loss 0.010


## 所学的卷积核的权重张量

In [29]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9827, -0.9799]])